## Imports

In [14]:
import pandas as pd
import sys
import os

In [15]:
# Use current working directory 
sys.path.insert(0, os.getcwd())
sys.path.append(os.path.abspath('..'))

### Import `src/scraper.py`

In [76]:
from src.scraper import(
    _clean_base,
    _fetch_with_requests,
    _incentive_score,
    _extract_internal_links,
    _fetch_raw_html,
    scrape_venue_pages,
    scrape_wayback,
    fallback_search,
    INCENTIVE_PATHS,
)

### Import `src/trainmodel.py`

In [ ]:
from src.trainmodel import load_dataset

In [50]:
# import run_model_pipeline
from run_model_pipeline import(
    run,
    load_all_venues,
    OUTPUT_DIR,  # data/model_output
    OUTPUT_FILE, # model_venues.json
    DEFAULT_SOURCE, # data/processed/json_batches_combined_presplit.json

    venue_to_place,
)

### Replicate: run() 
* `run_model_pipeline.py` -> run()
* Parameters: `run(5)`
    * indices
    * offset
    * limit
    * source
    * output

In [80]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_file = OUTPUT_FILE # output or 

all_venues = load_all_venues(DEFAULT_SOURCE)
# all_venues

In [27]:
df = load_dataset()
df.head()
# df["source"].value_counts()
# df["category"].value_counts()
# df[["text", "category", "btype", "source"]].sample(10)

,text,category,motivator,cuisine,btype,source
0,Whether youre celebrating something special or...,Discount,Value,Other,Live Music,presplit
1,"Try one of the best hot dogs you'll ever eat, ...",Free,Free,Other,Nightclub,presplit
2,BRUNCH SATURDAY 11:00am to 3:00pm SUNDAY 10:00...,Live Music,Social,Live Music,Live Music,presplit
3,Join us from 3-6pm Wednesday-Sunday for $9 sig...,Happy Hour,Social,Other,Nightclub,presplit
4,"No Cover charge before 9:30pm on weekends, exc...",Free,Free,Other,Nightclub,presplit


## trainmodel.py

### `load_all_venues` doesn't do shit

In [82]:
import json

from json_extract import(
    output_model_venue,
)

In [83]:
model_venue = output_model_venue()
# Read JSON file from Piolo's scraped data:
# try: 
#     with open(DEFAULT_SOURCE, 'r') as file:
#         model_venue = json.load(file)
#     print(json.dumps(model_venue, indent=4))
# except FileNotFoundError:
#     print("Error: The json file was not found.")
# type(model_venue)

[
    {
        "venue_id": "ChIJaRNDj6L2woARhpdaMEe_Mm8",
        "row": 1,
        "venue_name": "49er Saloon",
        "address": "31908 Crown Valley Road, Acton, CA",
        "city": "Acton",
        "state": "CA",
        "Business Type": "Bar",
        "Cuisine / Experience Category": "Restaurant",
        "Incentive Category": "No Incentive",
        "Incentive Teaser": "No incentive found",
        "Full Incentive Description": "No incentive found",
        "Days / Timing Restrictions": "Unknown",
        "Group Friendly?": "Unknown",
        "Psychological Motivator Type": "Unknown",
        "Estimated Perceived Value ($ range)": "Unknown",
        "Expiration / Ongoing": "Unknown",
        "Source URL": "https://www.49ersaloon.com/",
        "Notes": "Could not scrape source",
        "_meta": {
            "model_confidence": 0.0,
            "scrape_time_s": 3.64,
            "inference_time_s": 0.27,
            "text_chars": 0
        }
    },
    {
        "venue_id": "C

In [42]:
def json_to_csv(json_object:list) -> pd.DataFrame:
    ls_teaser = []
    ls_full = []
    for element in json_object:
        ls_teaser.append(element["Incentive Teaser"])
        ls_full.append(element["Full Incentive Description"])

    df = pd.DataFrame({'Incentive Teaser' : ls_teaser, 'Full Incentive Description' : ls_full})
    return df

In [43]:
df = json_to_csv(model_venue)
df

,Incentive Teaser,Full Incentive Description
0,No incentive found,No incentive found
1,Whether you’re celebrating something special o...,Whether you’re celebrating something special o...
2,"Try one of the best hot dogs you'll ever eat, ...","Try one of the best hot dogs you'll ever eat, ..."
3,BRUNCH SATURDAY 11:00am to 3:00pm SUNDAY 10:00...,BRUNCH SATURDAY 11:00am to 3:00pm SUNDAY 10:00...
4,Join us from 3-6pm Wednesday-Sunday for $9 sig...,Join us from 3-6pm Wednesday-Sunday for $9 sig...
...,...,...
95,No incentive found,No incentive found
96,No incentive found,No incentive found
97,No incentive found,No incentive found
98,"By providing your phone number, you agree to r...","By providing your phone number, you agree to r..."


In [37]:
def load_all_venues(path=DEFAULT_SOURCE):
    with open(path) as f:
        content = f.read()

    # Handle both plain JSON arrays and concatenated JSON objects
    try:
        data = json.loads(content)
        venues = data if isinstance(data, list) else [data]
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        pos, venues = 0, []
        while pos < len(content):
            stripped = content[pos:].lstrip()
            if not stripped:
                break
            skip = len(content[pos:]) - len(stripped)
            try:
                obj, end = decoder.raw_decode(stripped)
                pos = pos + skip + end
                if isinstance(obj, list):
                    venues.extend(v for v in obj if isinstance(v, dict))
                elif isinstance(obj, dict):
                    venues.append(obj)
            except Exception:
                break

    return [v for v in venues if v.get("Source URL")]

In [ ]:
# Check if model_venues & load_venues (from `load_all_venues()`) are the same
# Answer: yes, they are
# NOTE: They are NOT the exact same. Apparently you can apply .get() to load_venues
# df1 = json_to_csv(load_venues)

In [77]:

load_venues = load_all_venues(DEFAULT_SOURCE)
load_venues = load_venues[0:0 + 10]
n_total = len(load_venues)
data_list = []
for i, load_venues in enumerate(load_venues):
    name = load_venues.get("venue_name", "Unknown")
    url = load_venues.get("Source URL", "")
    place = venue_to_place(load_venues)

    print(f"[{i}/{n_total}] {name}")
    print(f"        {url}")

    btype = load_venues.get("Business Type", "")
    text = scrape_venue_pages(url, business_type=btype)
    if not text:
        text = scrape_wayback(url)
        if text:
            scrape_source = "wayback"
    if not text:
        text = fallback_search(name, load_venues.get("city", ""))
        if text:
            scrape_source = "serper_fallback"
    data_list.append(text)
df2 = pd.DataFrame(data_list)
    

[0/10] 49er Saloon
        https://www.49ersaloon.com/
Scraping: https://www.49ersaloon.com/happy-hour
Scraping: https://www.49ersaloon.com/happyhour
Scraping: https://www.49ersaloon.com/specials
Scraping: https://www.49ersaloon.com/deals
Scraping: https://www.49ersaloon.com/entertainment
Scraping: https://www.49ersaloon.com/live-music
Scraping: https://www.49ersaloon.com/events
Scraping: https://www.49ersaloon.com/menu
Scraping: https://www.49ersaloon.com
  -> JS homepage for link discovery...
  -> deep type paths (Bar): ['/drink-specials', '/bar-specials', '/pricing', '/drinks-menu']
  -> JS renderer for 4 sparse path(s)...
  -> Wayback (/) 80808013848/http://49ersaloon.com/...
  -> Serper fallback: "49er Saloon" Acton happy hour specials deals
[1/10] 26 Beach
        https://26beach.com/
Scraping: https://26beach.com/happy-hour
Scraping: https://26beach.com/happyhour
Scraping: https://26beach.com/specials
Scraping: https://26beach.com/deals
Scraping: https://26beach.com/entertainmen

In [78]:
df2

,0
0,Great day at the 49er Saloon in Acton! - Faceb...
1,Get Ready to Treat Yourself Whether you’re cel...
2,top of page 2J'S Lounge 120 W. HOUSTON AVE. ...
3,top of page Video by Elevated Aerial Photograp...
4,Reservations Happy Hour at 333 Pacific Happy H...
5,
6,Food • Bar • Entertainment • Dance Join us Wed...
7,Savor True Mexican Taste Dive into Aguachile a...
8,El 43 Bar & Nightclub 630 F st Wasco Ca 93280 ...
9,
